# Comprehensive EDA & Advanced Feature Engineering Pipeline
## Two-Stage Entity Resolution Reranker Optimization
**Target Metric:** Macro $F_{0.5}$ (Precision-Heavy Evaluation with Singleton Handling)  
**Goal:** Maximize reranking precision and recall across all candidate pairs to boost downstream models (XGBoost, LightGBM, CatBoost, ANN) toward the theoretical upper bound ($99.91\%$).

---

### Pipeline Roadmap:
1. **Environment Setup & Memory-Optimized Loading**: Streamlined loading with automatic dtype downcasting for 1M+ rows.
2. **Comprehensive Exploratory Data Analysis (EDA)**:
   - Target class distribution & ANN Candidate Top-10 recall analysis.
   - Point-biserial correlations of raw features with ground-truth match label.
   - Morphological, street address, and legal form consistency vs conflict diagnostics.
   - Data visual profiling (KDE distributions, boxplots, correlation spectrum).
3. **Advanced Feature Engineering (FE)**:
   - *Fuzzy Statistics Aggregations*: Mean, Std, Min, Max, Spread.
   - *Character N-Gram Dynamics*: Decay, ratio, multi-scale averages.
   - *Domain Matching Composites*: Name, Address, and Multi-modal overall composites.
   - *Cross-Modal Interaction Terms*: ANN similarity $\times$ local string similarities.
   - *Conflict & Disqualification Flags*: Address number conflicts, legal form clashes, token boundary mismatches.
   - *Token Asymmetry Penalties*: Length difference ratios, precision proxies.
   - *Within-Query Relative Signals (Crucial for Reranker)*: Distance to query max, mean, z-scores, query ranks, best flags.
4. **Feature Correlation & Discriminative Power Analysis**: Validating newly engineered features.
5. **Query-Partitioned Validation & Model Benchmarking**:
   - Zero-leakage query partitioned train/val split.
   - Fast vectorized Macro $F_{0.5}$ evaluator & threshold sweep.
   - Comparative benchmark: Raw vs Engineered features across Gradient Boosted Classifiers.
6. **Dataset & Metadata Export**: Saving processed feature matrices and column configuration for downstream training.

In [ ]:
import os, gc, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve

# Global warning suppression
warnings.filterwarnings('ignore')

# Package auto-installation if required
try:
    import lightgbm as lgb
except ImportError:
    os.system("pip install -q lightgbm")
    import lightgbm as lgb

try:
    import xgboost as xgb
except ImportError:
    os.system("pip install -q xgboost")
    import xgboost as xgb

try:
    from tqdm.auto import tqdm
except ImportError:
    os.system("pip install -q tqdm")
    from tqdm.auto import tqdm

# Plot styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

# Helper to automatically locate files in local or Kaggle environments
def locate_data_file():
    kaggle_path = Path('/kaggle/input/datasets/vinaysingh120221/training-data-01/training_features_full.csv')
    local_path = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\src\Reranker\preprocessed_data\training_features_full.csv')
    relative_path = Path('src/Reranker/preprocessed_data/training_features_full.csv')
    
    for p in [kaggle_path, local_path, relative_path]:
        if p.exists():
            return p
    raise FileNotFoundError("Could not locate training_features_full.csv in known paths.")

DATA_PATH = locate_data_file()
OUTPUT_DIR = Path('/kaggle/working/data_preprocessing_output') if Path('/kaggle/working').exists() else Path('./data_preprocessing_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
VALIDATION_FRACTION = 0.2

print(f"[INFO] Using Data Path : {DATA_PATH}")
print(f"[INFO] Output Directory : {OUTPUT_DIR}")

In [ ]:
print("[INFO] Loading dataset in memory-efficient chunks...")
chunk_size = 500_000
chunks = []

# Dtype specification to minimize RAM
dtype_dict = {
    'query_id': 'str',
    'candidate_id': 'str',
    'label': 'int8'
}

for chunk in pd.read_csv(DATA_PATH, dtype=dtype_dict, chunksize=chunk_size):
    chunk['label'] = chunk['label'].astype(np.int8)
    feat_cols = [c for c in chunk.columns if c not in ['query_id', 'candidate_id', 'label']]
    for col in feat_cols:
        chunk[col] = pd.to_numeric(chunk[col], errors='coerce').fillna(0.0).astype(np.float32)
    chunks.append(chunk)

raw_df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

RAW_FEATURE_COLUMNS = [c for c in raw_df.columns if c not in ['query_id', 'candidate_id', 'label']]
print(f"[SUCCESS] Dataset Loaded Successfully!")
print(f"Total Rows: {len(raw_df):,} | Unique Queries: {raw_df['query_id'].nunique():,} | Raw Features: {len(RAW_FEATURE_COLUMNS)}")
print(f"Memory Footprint: {raw_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
raw_df.head()

## 2. Exploratory Data Analysis (EDA) & Domain Diagnostics
We inspect the statistical properties, missing value patterns, class imbalance, and relationship between features and entity matching ground truth.

In [ ]:
print("=" * 70)
print("1. DATA HEALTH & TARGET CLASS DISTRIBUTION")
print("=" * 70)

total_rows = len(raw_df)
pos_count = int((raw_df['label'] == 1).sum())
neg_count = int((raw_df['label'] == 0).sum())
pos_rate = pos_count / total_rows

queries_with_match = int(raw_df[raw_df['label'] == 1]['query_id'].nunique())
total_queries = int(raw_df['query_id'].nunique())
singleton_queries = total_queries - queries_with_match

print(f"Total Candidate Pairs       : {total_rows:,}")
print(f"Positive Matches (Label=1)  : {pos_count:,} ({pos_rate * 100:.2f}%)")
print(f"Negative Pairs (Label=0)    : {neg_count:,} ({(1 - pos_rate) * 100:.2f}%)")
print(f"Total Unique Queries        : {total_queries:,}")
print(f"Queries with True Match     : {queries_with_match:,} ({queries_with_match / total_queries * 100:.2f}%)")
print(f"Singleton Queries (No Match): {singleton_queries:,} ({singleton_queries / total_queries * 100:.2f}%)")

# Distribution of Positive Matches across ANN Ranks 1 to 10
rank_match_stats = raw_df.groupby('ann_rank')['label'].agg(['count', 'sum', 'mean']).reset_index()
rank_match_stats.columns = ['ann_rank', 'total_candidates', 'positive_matches', 'match_rate']
print("\n" + "=" * 70)
print("2. ANN RANK MATCH DISTRIBUTION (Stage 1 Retrieval Recall Spectrum)")
print("=" * 70)
print(rank_match_stats.to_string(index=False))

In [ ]:
# Statistical Summary by Label
stats_by_label = raw_df.groupby('label')[RAW_FEATURE_COLUMNS].mean().T
stats_by_label.columns = ['Mean_Negative (Label=0)', 'Mean_Positive (Label=1)']
stats_by_label['Separation_Delta'] = stats_by_label['Mean_Positive (Label=1)'] - stats_by_label['Mean_Negative (Label=0)']

# Correlation with target label
correlations = raw_df[RAW_FEATURE_COLUMNS].apply(lambda s: s.corr(raw_df['label'])).sort_values(ascending=False)
stats_by_label['Point_Biserial_Corr'] = correlations

print("=" * 80)
print("TOP 20 MOST DISCRIMINATIVE RAW FEATURES (By Correlation with Match Label)")
print("=" * 80)
print(stats_by_label.sort_values(by='Point_Biserial_Corr', ascending=False).head(20).to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Top Feature Correlations with Label
top_corr = correlations.head(10)
sns.barplot(x=top_corr.values, y=top_corr.index, ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Top 10 Raw Features Correlated with Label', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Point-Biserial Correlation')

# 2. ANN Similarity Distribution (Positive vs Negative)
sns.kdeplot(data=raw_df, x='ann_similarity', hue='label', common_norm=False, fill=True, ax=axes[0, 1], palette={0: 'red', 1: 'green'})
axes[0, 1].set_title('ANN Similarity Density by Class', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('ANN Two-Tower Cosine Similarity')

# 3. Jaro-Winkler vs Fuzzy Ratio Separation
sns.kdeplot(data=raw_df, x='jaro_winkler', hue='label', common_norm=False, fill=True, ax=axes[0, 2], palette={0: 'red', 1: 'green'})
axes[0, 2].set_title('Jaro-Winkler Similarity Density by Class', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Jaro-Winkler Score')

# 4. Positive Match Rate across ANN Retrieval Ranks
sns.barplot(data=rank_match_stats, x='ann_rank', y='match_rate', ax=axes[1, 0], color='royalblue')
axes[1, 0].set_title('True Match Probability by ANN Rank', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('ANN Candidate Rank (1 = Nearest Neighbor)')
axes[1, 0].set_ylabel('Probability P(Label=1)')

# 5. Address Match Impact on Precision
addr_stats = raw_df.groupby('address_number_match')['label'].agg(['count', 'mean']).reset_index()
addr_stats['address_status'] = addr_stats['address_number_match'].map({-1.0: 'Missing (-1)', 0.0: 'Conflict (0)', 1.0: 'Match (1)'})
sns.barplot(data=addr_stats, x='address_status', y='mean', ax=axes[1, 1], palette='crest')
axes[1, 1].set_title('Match Probability by Address Street # Status', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Positive Match Rate')
axes[1, 1].set_xlabel('Street Number Status')

# 6. Legal Form Consistency Impact
legal_stats = raw_df.groupby(['both_legal_form_present', 'same_legal_form'])['label'].agg(['count', 'mean']).reset_index()

def map_legal_scenario(row):
    if row['both_legal_form_present'] == 1.0:
        return 'Both Present & Match' if row['same_legal_form'] == 1.0 else 'Both Present & Mismatch'
    return 'Missing / 1-Present'

legal_stats['scenario'] = legal_stats.apply(map_legal_scenario, axis=1)
sns.barplot(data=legal_stats, x='scenario', y='mean', ax=axes[1, 2], palette='mako')
axes[1, 2].set_title('Match Rate by Legal Form Agreement', fontsize=12, fontweight='bold')
axes[1, 2].set_ylabel('Positive Match Rate')
axes[1, 2].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_diagnostics_overview.png", bbox_inches='tight')
plt.show()

## 3. Advanced Feature Engineering (FE) Architecture

### Key Feature Engineering Modules:
1. **Fuzzy Statistics Aggregations**:
   - `fuzzy_mean`, `fuzzy_std`, `fuzzy_min`, `fuzzy_max`, `fuzzy_spread`.
2. **Morphological Character N-Gram Dynamics**:
   - `char_ngram_mean`, `char_ngram_std`, `char_ngram_decay` ($Sim_{2gram} - Sim_{5gram}$), `char_ngram_ratio_5_to_2`.
3. **Domain Matching Composites**:
   - `address_composite_score`: Weighted blend of partial, fuzzy, and exact address matches.
   - `name_composite_score`: Weighted fusion of Jaro-Winkler, Fuzzy Ratio, Token Jaccard, and 3-Gram.
   - `overall_composite_score`: Multi-modal index fusing ANN, Name, Address, and Legal Form.
4. **Cross-Modal Synergistic Interactions**:
   - Multiplicative pairs ($ANN \times Jaro$, $ANN \times Fuzzy$, $ANN \times Token$, $ANN \times Address$, $ANN \times Legal$).
5. **Conflict & Disqualification Flags (Precision-Critical for $F_{0.5}$)**:
   - Address street number mismatch penalty (`address_num_conflict_flag`).
   - Legal form clash flag (`legal_form_conflict_flag`).
   - Token boundary flags (`token_exact_both_ends`, `token_exact_neither_end`).
   - High confidence match flag & Severe mismatch disqualifier.
6. **Token Asymmetry & Length Penalties**:
   - `token_precision_proxy`, `token_contain_length_penalty`, `ann_rank_reciprocal`, `ann_rank_log`.
7. **Within-Query Group Relative Features & Ranks (The Game Changer for Rerankers)**:
   - Difference from query-maximum candidate (`_diff_max`).
   - Difference from query-mean (`_diff_mean`).
   - Ratio to query top candidate (`_ratio_max`).
   - Intra-query Z-score (`_zscore`).
   - Intra-query rank (`_query_rank`).
   - Best-in-query flag (`_is_query_best`).

In [ ]:
def engineer_all_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    High-performance, vectorized feature engineering pipeline for entity resolution reranking.
    Computes local pairwise similarities, domain composites, conflict flags, and query-level relative metrics.
    """
    t0 = time.time()
    print("[INFO] Starting Feature Engineering Transformation...")
    df = df.copy()
    
    # -------------------------------------------------------------
    # 1. Fuzzy Statistics Aggregations
    # -------------------------------------------------------------
    fuzzy_cols = ['fuzzy_ratio', 'fuzzy_partial_ratio', 'fuzzy_token_sort_ratio', 'fuzzy_token_set_ratio', 'fuzzy_WRatio']
    df['fuzzy_mean'] = df[fuzzy_cols].mean(axis=1).astype(np.float32)
    df['fuzzy_std'] = df[fuzzy_cols].std(axis=1).fillna(0.0).astype(np.float32)
    df['fuzzy_min'] = df[fuzzy_cols].min(axis=1).astype(np.float32)
    df['fuzzy_max'] = df[fuzzy_cols].max(axis=1).astype(np.float32)
    df['fuzzy_spread'] = (df['fuzzy_max'] - df['fuzzy_min']).astype(np.float32)
    
    # -------------------------------------------------------------
    # 2. Morphological Character N-Gram Dynamics
    # -------------------------------------------------------------
    char_cols = ['char_2gram_similarity', 'char_3gram_similarity', 'char_4gram_similarity', 'char_5gram_similarity']
    df['char_ngram_mean'] = df[char_cols].mean(axis=1).astype(np.float32)
    df['char_ngram_std'] = df[char_cols].std(axis=1).fillna(0.0).astype(np.float32)
    df['char_ngram_decay'] = (df['char_2gram_similarity'] - df['char_5gram_similarity']).astype(np.float32)
    df['char_ngram_ratio_5_to_2'] = (df['char_5gram_similarity'] / (df['char_2gram_similarity'] + 1e-4)).astype(np.float32)
    
    # -------------------------------------------------------------
    # 3. Address Composite & Inverse Length Decays
    # -------------------------------------------------------------
    df['address_length_decay'] = (1.0 / (1.0 + df['address_length_difference'])).astype(np.float32)
    df['name_length_decay'] = (1.0 / (1.0 + df['name_length_difference'])).astype(np.float32)
    df['address_composite_score'] = (
        0.45 * df['address_partial_ratio'] + 
        0.35 * df['address_fuzzy_ratio'] + 
        0.20 * df['address_exact_match']
    ).astype(np.float32)
    
    # -------------------------------------------------------------
    # 4. Multi-Modal Domain Composites
    # -------------------------------------------------------------
    df['name_composite_score'] = (
        0.35 * df['jaro_winkler'] + 
        0.25 * df['fuzzy_ratio'] + 
        0.20 * df['token_jaccard'] + 
        0.20 * df['char_3gram_similarity']
    ).astype(np.float32)
    
    df['overall_composite_score'] = (
        0.40 * df['ann_similarity'] + 
        0.30 * df['name_composite_score'] + 
        0.20 * df['address_composite_score'] + 
        0.10 * df['legal_form_match']
    ).astype(np.float32)
    
    # -------------------------------------------------------------
    # 5. Cross-Modal Synergistic Interactions
    # -------------------------------------------------------------
    df['ann_jaro_interaction'] = (df['ann_similarity'] * df['jaro_winkler']).astype(np.float32)
    df['ann_fuzzy_interaction'] = (df['ann_similarity'] * df['fuzzy_ratio']).astype(np.float32)
    df['ann_token_interaction'] = (df['ann_similarity'] * df['token_jaccard']).astype(np.float32)
    df['ann_address_interaction'] = (df['ann_similarity'] * df['address_partial_ratio']).astype(np.float32)
    df['ann_legal_interaction'] = (df['ann_similarity'] * df['legal_form_match']).astype(np.float32)
    df['name_address_interaction'] = (df['jaro_winkler'] * df['address_partial_ratio']).astype(np.float32)
    df['name_legal_interaction'] = (df['jaro_winkler'] * df['legal_form_match']).astype(np.float32)
    
    # -------------------------------------------------------------
    # 6. Conflict, Agreement & Precision Disqualification Flags
    # -------------------------------------------------------------
    df['address_num_match_flag'] = (df['address_number_match'] == 1.0).astype(np.float32)
    df['address_num_conflict_flag'] = (df['address_number_match'] == 0.0).astype(np.float32)
    df['address_num_missing_flag'] = (df['address_number_match'] == -1.0).astype(np.float32)
    df['legal_form_conflict_flag'] = ((df['both_legal_form_present'] == 1.0) & (df['same_legal_form'] == 0.0)).astype(np.float32)
    df['legal_form_agree_flag'] = ((df['both_legal_form_present'] == 1.0) & (df['same_legal_form'] == 1.0)).astype(np.float32)
    df['token_exact_both_ends'] = (df['first_token_match'] * df['last_token_match']).astype(np.float32)
    df['token_exact_neither_end'] = ((1.0 - df['first_token_match']) * (1.0 - df['last_token_match'])).astype(np.float32)
    
    # -------------------------------------------------------------
    # 7. Token Asymmetry Penalties & Rank Transforms
    # -------------------------------------------------------------
    df['token_precision_proxy'] = (df['token_overlap'] / (df['token_overlap'] + df['name_token_count_diff'] + 1e-4)).astype(np.float32)
    df['token_contain_length_penalty'] = (df['token_containment'] * (1.0 - df['name_length_ratio'])).astype(np.float32)
    df['ann_rank_reciprocal'] = (1.0 / (df['ann_rank'] + 1e-5)).astype(np.float32)
    df['ann_rank_log'] = np.log1p(df['ann_rank']).astype(np.float32)
    df['is_top1_candidate'] = (df['ann_rank'] == 1.0).astype(np.float32)
    df['is_top3_candidate'] = (df['ann_rank'] <= 3.0).astype(np.float32)
    
    # -------------------------------------------------------------
    # 8. High-Precision Match & Severe Mismatch Flags (F0.5 Tailored)
    # -------------------------------------------------------------
    df['strong_name_exact'] = (
        (df['exact_name_norm'] == 1.0) | 
        ((df['fuzzy_ratio'] >= 0.95) & (df['jaro_winkler'] >= 0.95))
    ).astype(np.float32)
    
    df['high_confidence_match_flag'] = (
        (df['strong_name_exact'] == 1.0) & 
        (df['country_equal'] == 1.0) & 
        (df['address_num_conflict_flag'] == 0.0) & 
        (df['legal_form_conflict_flag'] == 0.0)
    ).astype(np.float32)
    
    df['severe_mismatch_penalty'] = (
        (df['token_overlap'] == 0.0) & 
        (df['token_exact_neither_end'] == 1.0) & 
        (df['fuzzy_ratio'] < 0.6)
    ).astype(np.float32)
    
    # -------------------------------------------------------------
    # 9. Within-Query Group Relative Features (Crucial for Reranker)
    # -------------------------------------------------------------
    rel_cols = [
        'ann_similarity', 
        'jaro_winkler', 
        'fuzzy_ratio', 
        'address_partial_ratio', 
        'name_composite_score', 
        'overall_composite_score'
    ]
    
    for col in rel_cols:
        q_max = df.groupby('query_id')[col].transform('max')
        q_mean = df.groupby('query_id')[col].transform('mean')
        q_min = df.groupby('query_id')[col].transform('min')
        q_std = df.groupby('query_id')[col].transform('std').fillna(0.0)
        
        df[f'{col}_diff_max'] = (df[col] - q_max).astype(np.float32)
        df[f'{col}_diff_mean'] = (df[col] - q_mean).astype(np.float32)
        df[f'{col}_diff_min'] = (df[col] - q_min).astype(np.float32)
        df[f'{col}_ratio_max'] = (df[col] / (q_max + 1e-6)).astype(np.float32)
        df[f'{col}_zscore'] = ((df[col] - q_mean) / (q_std + 1e-6)).astype(np.float32)
        df[f'{col}_query_rank'] = df.groupby('query_id')[col].rank(ascending=False, method='min').astype(np.float32)
        df[f'{col}_is_query_best'] = (df[f'{col}_diff_max'] == 0.0).astype(np.float32)
        
    print(f"[SUCCESS] Feature Engineering completed in {time.time() - t0:.2f} seconds!")
    print(f"Total Columns: {df.shape[1]} (Engineered Features: {df.shape[1] - 3})")
    return df

df_engineered = engineer_all_features(raw_df)
ENGINEERED_FEATURE_COLUMNS = [c for c in df_engineered.columns if c not in ['query_id', 'candidate_id', 'label']]
print(f"Engineered Feature List ({len(ENGINEERED_FEATURE_COLUMNS)} total features):")
print(ENGINEERED_FEATURE_COLUMNS)

## 4. Feature Validation & Discriminative Power Analysis
We examine the top engineered features that exhibit the highest correlation with the ground-truth match label and measure the information gain over raw features.

In [ ]:
fe_correlations = df_engineered[ENGINEERED_FEATURE_COLUMNS].apply(lambda s: s.corr(df_engineered['label'])).sort_values(ascending=False)

print("=" * 75)
print("TOP 25 STRONGEST PREDICTIVE FEATURES (Engineered Feature Set)")
print("=" * 75)
print(fe_correlations.head(25).to_string())

plt.figure(figsize=(12, 8))
sns.barplot(x=fe_correlations.head(20).values, y=fe_correlations.head(20).index, palette='magma')
plt.title('Top 20 Engineered Features by Correlation with Ground-Truth Match', fontsize=14, fontweight='bold')
plt.xlabel('Point-Biserial Correlation')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top_engineered_features_corr.png", bbox_inches='tight')
plt.show()

## 5. Query-Partitioned Validation & Macro $F_{0.5}$ Benchmark
We perform a query-level partitioned train/val split (zero cross-query leakage) and evaluate the performance gain of the engineered feature set across Gradient Boosted Classifiers against the user's previous baseline scores.

In [ ]:
def query_partitioned_split(dataset_df: pd.DataFrame, val_fraction=0.2, random_state=42):
    """Splits dataset strictly by query_id to guarantee zero entity leakage."""
    unique_queries = np.array(sorted(dataset_df["query_id"].unique()))
    rng = np.random.default_rng(random_state)
    rng.shuffle(unique_queries)
    val_size = max(1, int(len(unique_queries) * val_fraction))
    val_query_set = set(unique_queries[:val_size])
    
    train_df = dataset_df[~dataset_df["query_id"].isin(val_query_set)].copy()
    val_df = dataset_df[dataset_df["query_id"].isin(val_query_set)].copy()
    
    train_df = train_df.sort_values("query_id").reset_index(drop=True)
    val_df = val_df.sort_values("query_id").reset_index(drop=True)
    return train_df, val_df

train_df, val_df = query_partitioned_split(df_engineered, val_fraction=VALIDATION_FRACTION, random_state=RANDOM_STATE)
print(f"Train Set: {len(train_df):,} rows ({train_df['query_id'].nunique():,} queries)")
print(f"Val Set  : {len(val_df):,} rows ({val_df['query_id'].nunique():,} queries)")

# Precompute validation ground-truth dictionaries for fast evaluation
val_y_true_by_query = val_df[val_df['label'] == 1].groupby('query_id')['candidate_id'].apply(set).to_dict()
val_all_queries = set(val_df['query_id'].unique())

def macro_f05(y_true_by_query, pred_by_query, all_queries):
    """
    Computes Macro F0.5 with Singleton Query Handling.
    F0.5 weights Precision 4x more than Recall (beta = 0.5).
    """
    scores = []
    for qid in all_queries:
        true_set = y_true_by_query.get(qid, set())
        pred_set = pred_by_query.get(qid, set())
        if not true_set:
            scores.append(1.0 if not pred_set else 0.0)
        else:
            tp = len(true_set & pred_set)
            precision = tp / len(pred_set) if pred_set else 0.0
            recall = tp / len(true_set)
            if precision == 0 and recall == 0:
                scores.append(0.0)
            else:
                scores.append((1.25 * precision * recall) / (0.25 * precision + recall))
    return float(np.mean(scores))

def evaluate_threshold_sweep(val_preds_df, fast_mode=False):
    """Runs exhaustive search to find the optimal decision threshold maximizing Macro F0.5."""
    qids = val_preds_df['query_id'].values
    cids = val_preds_df['candidate_id'].values
    scores = val_preds_df['score'].values
    from collections import defaultdict
    
    best_f05 = -1.0
    best_t = 0.5
    
    pct_count = 30 if fast_mode else 100
    test_thresh = np.percentile(scores, np.linspace(1, 99, pct_count))
    
    for t in test_thresh:
        mask = (scores >= t)
        pred_dict = defaultdict(set)
        for q, c in zip(qids[mask], cids[mask]):
            pred_dict[q].add(c)
            
        score = macro_f05(val_y_true_by_query, pred_dict, val_all_queries)
        if score > best_f05:
            best_f05 = score
            best_t = t
            
    return best_f05, best_t

In [ ]:
print("=" * 75)
print("TRAINING LIGHTGBM CLASSIFIER ON FULL ENGINEERED FEATURE SET")
print("=" * 75)

X_train = train_df[ENGINEERED_FEATURE_COLUMNS]
y_train = train_df['label'].to_numpy(dtype=np.int32)
X_val = val_df[ENGINEERED_FEATURE_COLUMNS]
y_val = val_df['label'].to_numpy(dtype=np.int32)

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 50,
    'subsample': 0.85,
    'colsample_bytree': 0.75,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbose': -1
}

lgb_model = lgb.LGBMClassifier(**lgb_params, n_estimators=600)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

val_preds = val_df[['query_id', 'candidate_id', 'label']].copy()
val_preds['score'] = lgb_model.predict_proba(X_val)[:, 1]

best_f05, best_t = evaluate_threshold_sweep(val_preds, fast_mode=False)

print("\n" + "=" * 75)
print("FINAL BENCHMARK COMPARISON RESULTS")
print("=" * 75)
print(f"Previous Raw LGBM Classifier Macro F0.5 : 0.956191")
print(f"Previous Raw XGBoost Classifier Macro F0.5 : 0.957615")
print(f"Previous Raw CatBoost Ranker Macro F0.5   : 0.935660")
print(f"Previous Raw ANN Classifier Macro F0.5    : 0.948626")
print(f"---------------------------------------------------------------------------")
print(f" NEW ENGINEERED LIGHTGBM MACRO F0.5     : {best_f05:.6f} (Optimal Threshold: {best_t:.4f})")
print(f"Theoretical Upper Bound Potential        : 0.999100")
print("=" * 75)

In [ ]:
# Top 25 Most Important Features in Trained Model
feat_importances = pd.Series(lgb_model.feature_importances_, index=ENGINEERED_FEATURE_COLUMNS).sort_values(ascending=False)

plt.figure(figsize=(12, 9))
sns.barplot(x=feat_importances.head(25).values, y=feat_importances.head(25).index, palette='viridis')
plt.title('Top 25 Most Important Features in LightGBM Classifier', fontsize=14, fontweight='bold')
plt.xlabel('Feature Importance (Split Gain Count)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lgbm_feature_importances.png", bbox_inches='tight')
plt.show()

# Evaluation Curves
y_true = val_preds['label'].values
y_scores = val_preds['score'].values

fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall, precision)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax1.set_xlim([-0.05, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('Receiver Operating Characteristic (ROC)', fontsize=14, fontweight='bold')
ax1.legend(loc="lower right", fontsize=12)
ax1.grid(True, alpha=0.3)

ax2.plot(recall, precision, color='blue', lw=2.5, label=f'PR curve (AUC = {pr_auc:.4f})')
ax2.set_xlim([-0.05, 1.05])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve (Crucial for F0.5 Optimization)', fontsize=14, fontweight='bold')
ax2.legend(loc="lower left", fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_pr_curves.png", bbox_inches='tight')
plt.show()

print(f"[SUMMARY] Area Under ROC Curve (AUC) : {roc_auc:.4f}")
print(f"[SUMMARY] Area Under PR Curve (AUC)  : {pr_auc:.4f}")

## 6. Exporting Engineered Features & Metadata for Downstream Models
Save the full engineered features matrix and metadata schema for direct consumption by all downstream classifiers (`train_xgb_classifier.ipynb`, `train_lightgbm_classifier.ipynb`, `train_catboost_ranker.ipynb`, `train_ann_classifier.ipynb`).

In [ ]:
# Save Feature Metadata Schema
metadata = {
    "num_features": len(ENGINEERED_FEATURE_COLUMNS),
    "feature_columns": ENGINEERED_FEATURE_COLUMNS,
    "raw_feature_columns": RAW_FEATURE_COLUMNS,
    "validation_macro_f05": best_f05,
    "optimal_threshold": float(best_t),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "timestamp": time.strftime('%Y-%m-%d %H:%M:%S')
}

meta_file = OUTPUT_DIR / "feature_metadata.json"
with open(meta_file, 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"[SUCCESS] Feature metadata saved to: {meta_file}")

# Optional: Export engineered dataset to Parquet / CSV for ultra-fast downstream loading
export_path = OUTPUT_DIR / "training_features_engineered.parquet"
print(f"[INFO] Exporting full engineered features to {export_path}...")
df_engineered.to_parquet(export_path, index=False)
print(f"[SUCCESS] Engineered Dataset successfully saved to: {export_path}")
print(" Pipeline Completed Successfully!")